In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
fut = pd.read_csv("./cache/fut.csv", )
cash_bond = pd.read_csv("./cache/cash_btp.csv")

fut['Date-Time'] = pd.to_datetime(fut['Date-Time']).dt.tz_localize(None)
fut = fut.set_index('Date-Time')
cash_bond['timestamp'] = pd.to_datetime(cash_bond['timestamp']).dt.tz_localize(None)
cash_bond = cash_bond.set_index('timestamp')

In [ ]:
rescaled_btp = MinMaxScaler().fit_transform(cash_bond['mid'].values.reshape(-1, 1))
rescaled_future = MinMaxScaler().fit_transform(fut['mid'].values.reshape(-1, 1))
plt.plot(rescaled_btp, label="mid-price Cash BTP")
plt.plot(rescaled_future, label="mid-price Future")
plt.legend()
plt.show()

In [ ]:
fut.index

In [ ]:
fut_mid = fut[['mid']]
cash_mid = cash_bond[['mid']]

In [ ]:
fut_mid = fut_mid.rename(columns = {"mid": "future_mid"})
cash_mid = cash_mid.rename(columns = {"mid": "cash_mid"})

In [ ]:
fut_mid

In [ ]:
aligned = cash_mid.join(fut_mid, how='inner').dropna()

In [ ]:
delta_number = 30

In [ ]:
aligned['delta_mid_fut_0'] = np.log(aligned['future_mid']/ aligned['future_mid'].shift(10))
aligned['target_fut'] = aligned['delta_mid_fut_0'].shift(-60) #target next 60 min
aligned['delta_mid_cash_0'] = np.log(aligned['cash_mid'] / aligned['cash_mid'].shift(10))
aligned['target_cash'] = aligned['delta_mid_cash_0'].shift(-60)

for i in range(delta_number):
    aligned['delta_mid_fut_{}'.format(i)] = aligned['delta_mid_fut_0'].shift(i)
    aligned['delta_mid_cash_{}'.format(i)] = aligned['delta_mid_cash_0'].shift(i)

In [ ]:
corrs = []
for lag in range(delta_number):  # test lags 1 to 5
    corr = aligned[f"delta_mid_cash_{lag}"].corr(aligned["target_fut"])
    corrs.append(corr)

plt.figure(figsize=(8,4))
plt.bar(range(1, delta_number + 1), corrs)
plt.xlabel("Lag")
plt.ylabel("Correlation")
plt.title("Lagged Return Correlation (Cash Bond leads Futures)")
plt.show()

In [ ]:
corrs = []
for lag in range(delta_number):  # test lags 1 to 5
    corr = aligned[f"delta_mid_fut_{lag}"].corr(aligned['target_cash'])
    corrs.append(corr)
plt.figure(figsize=(8,4))
plt.bar(range(1, delta_number + 1), corrs)
plt.xlabel("Lag")
plt.ylabel("Correlation")
plt.title("Lagged Return Correlation (Futures leads Cash Bond)")
plt.show()


In [ ]:
corrs = []
for lag in range(delta_number):  # test lags 1 to 5
    corr = aligned[f"delta_mid_fut_{lag}"].corr(aligned['target_fut'])
    corrs.append(corr)
plt.figure(figsize=(8,4))
plt.bar(range(1, delta_number + 1), corrs)
plt.xlabel("Lag")
plt.ylabel("Correlation")
plt.title("Lagged Return Correlation Classic Setting (Futures only)*")
plt.show()

In [ ]:
import seaborn as sns

corr_matrix = aligned[[f'delta_mid_cash_{lag}' for lag in range(delta_number)] + ['target_fut']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
plt.title("Lagged Returns Correlation Heatmap")
plt.show()

In [ ]:
from statsmodels.tsa.stattools import coint

# Run cointegration test
coint_test = coint(aligned['future_mid'], aligned['cash_mid'])

# Extract test statistic & p-values
test_statistic = coint_test[0]
p_value = coint_test[1]
conf_90 = coint_test[2][0] # 90% confidence interval critical value
conf_95 = coint_test[2][1] # 95% confidence interval critical value

# Determine if the assets are cointegrated at the 90th & 95th confidence intervals
cointegrated_90 = test_statistic < conf_90
cointegrated_95 = test_statistic < conf_95

test_statistic, p_value, conf_90, conf_95, cointegrated_90, cointegrated_95

In [ ]:
def rolling_cointegration(data1, data2, window_length):
    """
    This function calculates rolling cointegration scores between
    two data series.

    Returns:
    pd.DataFrame: A DataFrame containing the rolling cointegration
    scores and p-values.
    """

    # Check if data1 and data2 have the same length
    if len(data1) != len(data2):
        raise ValueError("The data series must be of the same length.")

    # Check if the length of the time window is valid
    if len(data1) < window_length or len(data2) < window_length:
        raise ValueError("The length of the time window is greater than \
        the available data series length.")

    rolling_coint_scores = []
    rolling_p_values = []

    for end in range(window_length, len(data1)):
        start = end - window_length
        series1 = data1[start:end]
        series2 = data2[start:end]

        score, p_value, _ = coint(series1, series2)

        rolling_coint_scores.append(score)
        rolling_p_values.append(p_value)

    results = pd.DataFrame({
        'coint_score': rolling_coint_scores,
        'p_value': rolling_p_values
    })

    return results

rolling_cointegration(aligned['future_mid'], aligned['cash_mid'], window_length=100)['p_value'].plot()